# 10 · Whole scenes of every split

The deployed models of both rounds, the slimmed EfficientNet-B0 U-Net and the teacher run on every HEO scene used anywhere in this work, at 2.5, 5 and 10 m, and on whole PhiSat-2 validation and test scenes (the six of each with the most fresh burn) at 4.75 m. Each scene is marked with its role:

- **train**: the networks learnt from it, so its numbers show how well they fit, not how they generalise;
- **search**: the scenes the search scored candidates on;
- **val**: the scenes the finalists and seeds were chosen on;
- **test**: read once per round in 08 and 09; this notebook only runs after both.

Nothing is chosen here: the deployed model is the one the round-2 rule picked. The figures and tables are for looking at where the models work and where they fail. Here the input is prepared as on board: only the image's own no-data pixels are set to zero (see the check below). The only Sentinel-2 data we have are the three test chips, already shown in 08 and 09.

The windows are written once to `data/pool_v2/review/` by the same functions as the scoring sets (about 13 GB, which can be deleted afterwards); figures and the per-scene table go to `models/pynas/review/`.

In [ ]:
# [1] setup
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                # the notebooks run from the repository root
import os, sys, json, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import Image, display
for p in (os.path.join(os.getcwd(), "pyqnas", "src"), os.getcwd()):
    if p not in sys.path:
        sys.path.insert(0, p)
import pynas
from pynas import config, final, report, runner, teacher
from pynas.train import data as D
from pynas.train.metrics import SCENE_MIN_PX
import burnscar.dataset as BD
import burnscar.pool as BP

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
cfg = config.load(Path("configs/burnscar.ini"))
spec = D.DataSpec.from_config(cfg)
dcfg = json.load(open("data/tables/dataset_config.json"))
scenes = pd.read_csv(dcfg["files"]["scenes"])
norm = json.load(open(dcfg["files"]["normalisation"]))
SIZE = dcfg["size"]
POOL, BASE = Path(spec.pool), Path(cfg.get("Run", "out"))
REVIEW, OUT = POOL / "review", BASE / "review"
REVIEW.mkdir(exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
CORES = len(os.sched_getaffinity(0))
dev = "cuda:0" if torch.cuda.is_available() else "cpu"
for f in ("models/onboard/burnscar_pynas/deploy.json", "models/onboard/burnscar_pynas_r2/deploy.json", BASE / "final_r2" / "test.json"):
    assert Path(f).exists(), f"{f} is missing: finish 08 and 09 (through the round-2 test) first"
print(f"PyNAS {pynas.__version__}; HEO pixel size in the scene table: {sorted(scenes[scenes.source == 'heo'].px_m.round(2).unique())} m")

In [ ]:
# [2] the models: deployed of both rounds, the slimmed EfficientNet-B0 U-Net (seed 0), the teacher (mean of its seeds)
R1, R2 = BASE / "final", BASE / "final_r2"
dep1 = json.load(open("models/onboard/burnscar_pynas/deploy.json"))
dep2 = json.load(open("models/onboard/burnscar_pynas_r2/deploy.json"))
res2 = final.results(R2 / "runs" / "queue.json")
eb0 = res2[(res2.label == "smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8") & (res2.seed == 0)].run.iloc[0]
models = {"PyNAS round 2": (runner.load_result(R2 / "runs" / dep2["run"], dev), "fp16"),
          "PyNAS round 1": (runner.load_result(R1 / "runs" / dep1["run"], dev), "fp16"),
          "U-Net EfficientNet-B0, slim": (runner.load_result(R2 / "runs" / eb0, dev), "fp16"),
          "teacher (mean of seeds)": (final.Mean(teacher.models(BASE / "teacher" / "runs" / "queue.json", dev)), "fp32")}
print({k: v[1] for k, v in models.items()})

## Input pixels without a label

In training and in every score so far, a pixel whose label is "no data" is also set to zero in the network input (`data.batch_input`, `input_mask="labels"`). Outside the image footprint that changes nothing. Inside it, real image values are hidden: the Sentinel-2 chips, for instance, leave a thin unlabelled band along every burn outline, so their input carries a zero line around each burn, which no image on board will have. First the share of pixels concerned in every set; then the deployed models and the slimmed U-Net scored with only the image's own no-data pixels set to zero (`input_mask="image"`, as on board), next to the scores as computed so far. Nothing is chosen from this; both versions are reported.

In [ ]:
# [3] pixels without a label: all of them, and those where the image has data (200 windows of every set)
rows = []
for name in [spec.train] + spec.search + spec.val + spec.test:
    s = D.open_set(spec, name, teacher=False)
    idx = np.unique(np.linspace(0, len(s["mask"]) - 1, min(len(s["mask"]), 200)).astype(int))
    m, r = np.asarray(s["mask"][idx]), np.asarray(s["refl"][idx])
    unlabelled, image = m == spec.nodata, (r != 0).any(1)
    rows.append(dict(set=name, unlabelled=unlabelled.mean(), unlabelled_inside_image=(unlabelled & image).mean()))
    del m, r
pd.DataFrame(rows).set_index("set").style.format("{:.2%}")

In [ ]:
# [4] fresh-burn IoU in FP16 (the teacher in FP32) with the input masked by the labels or by the image only
from pynas.train.metrics import evaluate, summary
t = spec.target_name
rows = []
for mname, (net, prec) in models.items():
    for name in spec.val + spec.test:
        s = D.open_set(spec, name, teacher=False)
        for im in ("labels", "image"):
            sc = summary(evaluate(net, s, spec, dev, prec, batch=16, input_mask=im), spec)
            rows.append(dict(model=mname, set=name, input_mask=im, pooled=sc[f"labels_{t}_iou"],
                             scene_mean=sc.get(f"labels_{t}_iou_scene_mean", np.nan)))
B = pd.DataFrame(rows)
B.to_csv(OUT / "input_mask_check.csv", index=False)
B.pivot_table(index=["set", "model"], columns="input_mask", values=["pooled", "scene_mean"]).round(3)

## The scenes and their windows

Every HEO scene with windows in the training pool or in a scoring set, with its role; the PhiSat-2 validation and test scenes with the most fresh burn in their scoring windows.

In [ ]:
# [5] the scenes, and their windows written once
def scenes_of(name):
    return set(pd.read_csv(POOL / f"{name}_windows.csv", usecols=["scene"]).scene)


role = {}
for name, r in [("heo_test", "test"), ("heo_val", "val"), ("heo_search", "search"), ("train", "train")]:
    for sc in scenes_of(name):
        role.setdefault(sc, r)
heo = scenes[(scenes.source == "heo") & scenes.scene.isin(role)].assign(role=lambda t: t.scene.map(role))
ph = []
for name, r in [("phisat2_val", "val"), ("phisat2_test", "test")]:
    w = pd.read_csv(POOL / f"{name}_windows.csv", usecols=["scene", "fresh"])
    top = w.groupby("scene").fresh.sum().sort_values(ascending=False).index[:6]
    ph.append(scenes[scenes.scene.isin(top)].assign(role=r))
ph = pd.concat(ph)
display(pd.concat([heo, ph]).groupby(["source", "role"]).scene.nunique().rename("scenes").to_frame())

SETS = {"heo_g2.5": (heo, [2.5]), "heo_g5": (heo, [5.0]), "heo_g10": (heo, [10.0]), "phisat2_g4.75": (ph, [4.75])}
for name, (rows, gsds) in SETS.items():
    prefix = REVIEW / name
    if Path(f"{prefix}_windows.csv").exists():
        continue
    w = BP.grid_windows(rows, gsds, SIZE).reset_index(drop=True)
    w["role"] = w.scene.map(dict(zip(rows.scene, rows.role)))
    print(f"{name}: {len(w):,} windows from {w.scene.nunique()} scenes")
    BP.write_set(BD.FixedWindows(w, scenes, norm, size=SIZE), str(prefix), workers=max(1, CORES - 1), every=0,
                 extra={"fresh": w.fresh.values, "old": w.old.values, "role": w.role.values})

In [ ]:
# [6] predictions (input masked by the image only, as on board), the per-scene table and the figures
# (four scenes a page: test first, then val, search, train)
ORDER = {"test": 0, "val": 1, "search": 2, "train": 3}
tables = []
for name in SETS:
    s = D.open_set(spec, f"review/{name}", teacher=False)
    preds = report.predict_set(models, spec, s, dev, input_mask="image")
    tables.append(report.scene_table(s, preds, spec).assign(set=name))
    lab = report.labels_of(s, spec)
    tags = dict(zip(s["windows"].scene, s["windows"].role))
    order = sorted(set(tags), key=lambda sc: (ORDER.get(tags[sc], 9), sc))
    for k in range(0, len(order), 4):
        fig = report.plot_scenes(s, preds, spec, name, scenes=order[k:k + 4], tags=tags, labels=lab)
        fig.savefig(OUT / f"{name}_{k // 4 + 1:02d}.png", dpi=150); plt.close(fig)
    print(f"{name}: {len(order)} scenes, {(len(order) + 3) // 4} pages")
    del s, preds, lab; gc.collect()
T = pd.concat(tables, ignore_index=True)
T.to_csv(OUT / "scene_iou.csv", index=False)

## Numbers

Mean fresh-burn IoU over the scenes of each role (scenes with at least 500 labelled fresh-burn pixels), then every scene. The train rows are fit, not generalisation; the gap between train and the other roles is the part worth reading.

In [ ]:
# [7] mean fresh-burn IoU per set and role
S = T[T.labelled_px >= SCENE_MIN_PX]
summary = S.pivot_table(index=["set", "role"], columns="model", values="iou", aggfunc="mean")
summary.insert(0, "scenes", S.groupby(["set", "role"]).scene.nunique())
summary.round(3)

In [ ]:
# [8] every scene: fresh-burn IoU, and false-alarm pixels in scenes with (almost) no labelled fresh burn
display(S.pivot_table(index=["set", "role", "scene"], columns="model", values="iou").round(2))
display(T[T.labelled_px < SCENE_MIN_PX].pivot_table(index=["set", "role", "scene"], columns="model", values="false_alarms"))

In [ ]:
# [9] the pages (all are in models/pynas/review/); here the first page of every set
for f in sorted(OUT.glob("*_01.png")):
    display(Image(filename=str(f)))